# CdpStC MA2020 GoC + CaHVA: NEURON vs BrainCell Cell.run

这个 notebook 在 GoC 的 `CdpStC_MA20_GoC.mod` 基础上插入 `CaHVA_MA20_GoC.mod`，检测 Ca channel 产生的 `ica` 是否能和 CDP/STC calcium-pool ion dynamics 一起工作。

重点对照：
- membrane voltage `v`
- `Ci/cai`
- `pump/pumpca` 与 `pump + pumpca` 守恒
- `CAM*` 状态
- CaHVA gates `s/u`
- CaHVA current：BrainCell current 对齐 `-NEURON ica`

运行前如果 NEURON 无法插入 `CaHVA_MA20_GoC`，在 `examples/neuron_compare/Cerebellum_mod/GoC` 下重编：

```bash
CPP=/usr/bin/cpp CC=/usr/bin/cc CXX=/usr/bin/c++ nrnivmodl ion/CdpStC_MA20_GoC.mod channel/CaHVA_MA20_GoC.mod
```

In [ ]:
import os
import sys
from pathlib import Path

repo_root = Path('/home/swl/braincell-ion_dyn').resolve()
if str(repo_root) in sys.path:
    sys.path.remove(str(repo_root))
sys.path.insert(0, str(repo_root))

os.environ.setdefault('JAX_PLATFORMS', 'cpu')

import brainstate
import brainunit as u
import matplotlib.pyplot as plt
import numpy as np

import braincell
from braincell import Branch, Cell, Morphology
from braincell.filter import BranchSlice, at
from braincell.mech import Channel, CurrentClamp, CurrentProbe, Ion, MechanismProbe, StateProbe

print('braincell version:', braincell.__version__)
print('braincell import path:', Path(braincell.__file__).resolve())

brainstate.environ.set(precision=64)


In [ ]:
repo_root = Path(braincell.__file__).resolve().parent.parent
mod_dir = repo_root / 'examples' / 'neuron_compare' / 'Cerebellum_mod' / 'GoC'
ion_mod = mod_dir / 'ion' / 'CdpStC_MA20_GoC.mod'
channel_mod = mod_dir / 'channel' / 'CaHVA_MA20_GoC.mod'
print('repo_root:', repo_root)
print('mod_dir:', mod_dir)
print('ion mod exists:', ion_mod.exists())
print('channel mod exists:', channel_mod.exists())

ion_text = ion_mod.read_text()
channel_text = channel_mod.read_text()
print('\nCdpStC INITIAL block preview:')
print(ion_text.split('INITIAL {', 1)[1].split('}', 1)[0][:1600])
print('\nCaHVA BREAKPOINT block preview:')
print(channel_text.split('BREAKPOINT {', 1)[1].split('}', 1)[0][:900])


In [ ]:
dt_ms = 0.05
duration_ms = 40.0
steps = int(duration_ms / dt_ms)
times_ms = np.arange(steps + 1) * dt_ms

temperature_celsius = 25.0
v_init_mV = -60.0
diam_um = 20.0
length_um = 20.0

# Keep the clamp weak. This minimal cell has no leak/Na/K balance, so a large
# current injection drives voltage unrealistically high and can destabilize the
# coupled ion/channel comparison.
stim_delay_ms = 5.0
stim_dur_ms = 20.0
stim_amp_nA = 0.005

gcabar_S_cm2 = 0.00046

cam_fields = [
    'CAM0',
    'CAM1C',
    'CAM2C',
    'CAM1N2C',
    'CAM1N',
    'CAM2N',
    'CAM2N1C',
    'CAM1C1N',
    'CAM4',
]

tracked_fields = ['v', 'Ci', 'ica', 's', 'u', 'pump', 'pumpca'] + cam_fields
print({
    'dt_ms': dt_ms,
    'duration_ms': duration_ms,
    'temperature_celsius': temperature_celsius,
    'v_init_mV': v_init_mV,
    'length_um': length_um,
    'diam_um': diam_um,
    'stim_delay_ms': stim_delay_ms,
    'stim_dur_ms': stim_dur_ms,
    'stim_amp_nA': stim_amp_nA,
    'gcabar_S_cm2': gcabar_S_cm2,
    'tracked_fields': tracked_fields,
})


In [ ]:
from neuron import h, load_mechanisms

compile_hint = (
    'Compile the GoC ion+channel mechanisms from examples/neuron_compare/Cerebellum_mod/GoC with:\n'
    'CPP=/usr/bin/cpp CC=/usr/bin/cc CXX=/usr/bin/c++ '
    'nrnivmodl ion/CdpStC_MA20_GoC.mod channel/CaHVA_MA20_GoC.mod'
)
if not load_mechanisms(str(mod_dir.resolve())):
    raise RuntimeError(f'NEURON mechanisms not found under {mod_dir!s}. {compile_hint}')
h.load_file('stdrun.hoc')

sec = h.Section(name='soma')
sec.L = length_um
sec.diam = diam_um
sec.nseg = 1
seg = sec(0.5)
try:
    sec.insert('CdpStC_MA20_GoC')
    sec.insert('CaHVA_MA20_GoC')
except ValueError as exc:
    raise RuntimeError(compile_hint) from exc

ion_mech = seg.CdpStC_MA20_GoC
channel_mech = seg.CaHVA_MA20_GoC
channel_mech.gcabar = gcabar_S_cm2

stim = h.IClamp(seg)
stim.delay = stim_delay_ms
stim.dur = stim_dur_ms
stim.amp = stim_amp_nA

h.celsius = temperature_celsius
h.dt = dt_ms
h.steps_per_ms = 1.0 / h.dt
h.tstop = duration_ms
h.v_init = v_init_mV

t_vec = h.Vector().record(h._ref_t)
v_vec = h.Vector().record(seg._ref_v)
cai_vec = h.Vector().record(seg._ref_cai)
ica_vec = h.Vector().record(seg._ref_ica)
pump_vec = h.Vector().record(ion_mech._ref_pump)
pumpca_vec = h.Vector().record(ion_mech._ref_pumpca)
s_vec = h.Vector().record(channel_mech._ref_s)
u_vec = h.Vector().record(channel_mech._ref_u)
cam_vectors = {name: h.Vector().record(getattr(ion_mech, f'_ref_{name}')) for name in cam_fields}

h.finitialize(h.v_init)
h.frecord_init()
h.continuerun(h.tstop)

neuron_t_ms = np.asarray(t_vec)
neuron_data = {
    'v': np.asarray(v_vec),
    'Ci': np.asarray(cai_vec),
    # BrainCell channels report inward Ca current as positive g*(E - V),
    # while this NMODL writes ica = g*(v - eca).
    'ica': -np.asarray(ica_vec),
    'pump': np.asarray(pump_vec),
    'pumpca': np.asarray(pumpca_vec),
    's': np.asarray(s_vec),
    'u': np.asarray(u_vec),
}
for name, vec in cam_vectors.items():
    neuron_data[name] = np.asarray(vec)
neuron_total = neuron_data['pump'] + neuron_data['pumpca']

print('max |-NEURON ica|:', float(np.max(np.abs(neuron_data['ica']))))
print('NEURON start/end v:', float(neuron_data['v'][0]), float(neuron_data['v'][-1]))
print('NEURON start/end cai:', float(neuron_data['Ci'][0]), float(neuron_data['Ci'][-1]))
print('NEURON start/end CaHVA s/u:', float(neuron_data['s'][0]), float(neuron_data['s'][-1]), float(neuron_data['u'][0]), float(neuron_data['u'][-1]))
print('NEURON start/end pump:', float(neuron_data['pump'][0]), float(neuron_data['pump'][-1]))
print('NEURON start/end pumpca:', float(neuron_data['pumpca'][0]), float(neuron_data['pumpca'][-1]))
print('NEURON max pump conserve drift:', float(np.max(np.abs(neuron_total - neuron_total[0]))))
for name in cam_fields:
    arr = neuron_data[name]
    print(f'NEURON {name} start/end:', float(arr[0]), float(arr[-1]))


In [ ]:
dt = dt_ms * u.ms
duration = duration_ms * u.ms

soma = Branch.from_lengths(lengths=[length_um] * u.um, radii=[diam_um / 2.0, diam_um / 2.0] * u.um, type='soma')
morpho = Morphology.from_root(soma, name='soma')
region = BranchSlice(branch_index=0, prox=0.0, dist=1.0)

cell = Cell(morpho, solver='staggered', V_init=v_init_mV * u.mV)
cell.paint(
    region,
    Ion(
        'CdpStC_MA2020_GoC',
        name='ca_stc',
        temp=u.celsius2kelvin(temperature_celsius),
    ),
)
cell.paint(
    region,
    Channel(
        'CaHVA_MA2020_GoC',
        ion_name='ca_stc',
        g_max=(gcabar_S_cm2 * 1000.0) * (u.mS / u.cm ** 2),
        temp=u.celsius2kelvin(temperature_celsius),
    ),
)
cell.place(
    at('soma', 0.5),
    CurrentClamp.step(stim_amp_nA * u.nA, stim_dur_ms * u.ms, delay=stim_delay_ms * u.ms),
    StateProbe(),
    MechanismProbe(mechanism='ca_stc', field='Ci'),
    MechanismProbe(mechanism='ca_stc', field='pump'),
    MechanismProbe(mechanism='ca_stc', field='pumpca'),
    MechanismProbe(mechanism='ca_stc', field='CAM0'),
    MechanismProbe(mechanism='ca_stc', field='CAM1C'),
    MechanismProbe(mechanism='ca_stc', field='CAM2C'),
    MechanismProbe(mechanism='ca_stc', field='CAM1N2C'),
    MechanismProbe(mechanism='ca_stc', field='CAM1N'),
    MechanismProbe(mechanism='ca_stc', field='CAM2N'),
    MechanismProbe(mechanism='ca_stc', field='CAM2N1C'),
    MechanismProbe(mechanism='ca_stc', field='CAM1C1N'),
    MechanismProbe(mechanism='ca_stc', field='CAM4'),
    MechanismProbe(mechanism='ca_stc', field='vrat'),
    MechanismProbe(mechanism='ca_stc', field='parea'),
    MechanismProbe(mechanism='ca_stc', field='dsq'),
    MechanismProbe(mechanism='ca_stc', field='dsqvol'),
    MechanismProbe(mechanism='CaHVA_MA2020_GoC', field='s'),
    MechanismProbe(mechanism='CaHVA_MA2020_GoC', field='u'),
    CurrentProbe(ion='ca_stc', mechanism='CaHVA_MA2020_GoC'),
)

with brainstate.environ.context(precision=64):
    cell.init_state()
    # HH channel reset_state sets gates to the NEURON INITIAL steady state.
    cell.reset_state()
    print('initial probe v (mV):', float(np.asarray(cell.sample_probe('soma(0.5)_v').to_decimal(u.mV)).reshape(-1)[0]))
    print('initial probe Ci (mM):', float(np.asarray(cell.sample_probe('soma(0.5)_ca_stc_Ci').to_decimal(u.mM)).reshape(-1)[0]))
    print('initial probe CaHVA s/u:', float(np.asarray(cell.sample_probe('soma(0.5)_CaHVA_MA2020_GoC_s')).reshape(-1)[0]), float(np.asarray(cell.sample_probe('soma(0.5)_CaHVA_MA2020_GoC_u')).reshape(-1)[0]))
    run_result = cell.run(dt=dt, duration=duration)

cell_data = {
    'v': np.asarray(run_result.traces['soma(0.5)_v'].to_decimal(u.mV)),
    'Ci': np.asarray(run_result.traces['soma(0.5)_ca_stc_Ci'].to_decimal(u.mM)),
    'pump': np.asarray(run_result.traces['soma(0.5)_ca_stc_pump'].to_decimal(u.mol / u.cm ** 2)),
    'pumpca': np.asarray(run_result.traces['soma(0.5)_ca_stc_pumpca'].to_decimal(u.mol / u.cm ** 2)),
    's': np.asarray(run_result.traces['soma(0.5)_CaHVA_MA2020_GoC_s']),
    'u': np.asarray(run_result.traces['soma(0.5)_CaHVA_MA2020_GoC_u']),
    'ica': np.asarray(run_result.traces['soma(0.5)_CaHVA_MA2020_GoC_current'].to_decimal(u.mA / (u.cm ** 2))),
}
for name in cam_fields:
    cell_data[name] = np.asarray(run_result.traces[f'soma(0.5)_ca_stc_{name}'].to_decimal(u.mM))
cell_geometry = {
    'vrat': np.asarray(run_result.traces['soma(0.5)_ca_stc_vrat']),
    'parea': np.asarray(run_result.traces['soma(0.5)_ca_stc_parea'].to_decimal(u.um)),
    'dsq': np.asarray(run_result.traces['soma(0.5)_ca_stc_dsq'].to_decimal(u.um ** 2)),
    'dsqvol': np.asarray(run_result.traces['soma(0.5)_ca_stc_dsqvol'].to_decimal(u.um ** 2)),
}
cell_total = cell_data['pump'] + cell_data['pumpca']

print('Cell.run max |current|:', float(np.max(np.abs(cell_data['ica']))))
print('Cell.run start/end v:', float(cell_data['v'][0]), float(cell_data['v'][-1]))
print('Cell.run start/end Ci:', float(cell_data['Ci'][0]), float(cell_data['Ci'][-1]))
print('Cell.run start/end CaHVA s/u:', float(cell_data['s'][0]), float(cell_data['s'][-1]), float(cell_data['u'][0]), float(cell_data['u'][-1]))
print('Cell.run start/end pump:', float(cell_data['pump'][0]), float(cell_data['pump'][-1]))
print('Cell.run start/end pumpca:', float(cell_data['pumpca'][0]), float(cell_data['pumpca'][-1]))
print('Cell.run max pump conserve drift:', float(np.max(np.abs(cell_total - cell_total[0]))))
for name in cam_fields:
    arr = cell_data[name]
    print(f'Cell.run {name} start/end:', float(arr[0]), float(arr[-1]))
for name, arr in cell_geometry.items():
    print(f'Cell.run {name} first/last:', float(arr[0]), float(arr[-1]))
for name, arr in cell_data.items():
    assert np.isfinite(arr).all(), f'{name} contains non-finite values'


In [ ]:
compare_t_ms = neuron_t_ms[1:]

def summarize_error(y, ref):
    y = np.asarray(y)
    ref = np.asarray(ref)
    n = min(len(y), len(ref))
    diff = y[:n] - ref[:n]
    return {
        'mae': float(np.mean(np.abs(diff))),
        'rmse': float(np.sqrt(np.mean(diff ** 2))),
        'max_abs': float(np.max(np.abs(diff))),
    }

error_summary = {}
for name in tracked_fields:
    neuron_cmp = neuron_data[name][1:]
    cell_cmp = cell_data[name]
    error_summary[name] = summarize_error(cell_cmp, neuron_cmp[:len(cell_cmp)])

fig, axes = plt.subplots(5, 1, figsize=(9, 18), sharex=True)

axes[0].plot(compare_t_ms, neuron_data['v'][1:], label='NEURON v')
axes[0].plot(compare_t_ms[:len(cell_data['v'])], cell_data['v'], '-.', label='BrainCell v')
axes[0].set_ylabel('v (mV)')
axes[0].legend()

axes[1].plot(compare_t_ms, neuron_data['Ci'][1:], label='NEURON cai')
axes[1].plot(compare_t_ms[:len(cell_data['Ci'])], cell_data['Ci'], '-.', label='BrainCell Ci')
axes[1].set_ylabel('Ci / cai (mM)')
axes[1].legend()

axes[2].plot(compare_t_ms, neuron_data['ica'][1:], label='-NEURON ica')
axes[2].plot(compare_t_ms[:len(cell_data['ica'])], cell_data['ica'], '-.', label='BrainCell CaHVA current')
axes[2].plot(compare_t_ms, neuron_data['s'][1:], ':', label='NEURON s')
axes[2].plot(compare_t_ms[:len(cell_data['s'])], cell_data['s'], '--', label='BrainCell s')
axes[2].plot(compare_t_ms, neuron_data['u'][1:], ':', label='NEURON u')
axes[2].plot(compare_t_ms[:len(cell_data['u'])], cell_data['u'], '--', label='BrainCell u')
axes[2].set_ylabel('CaHVA current / gates')
axes[2].legend(ncol=2, fontsize=8)

for name in ['CAM0', 'CAM1C', 'CAM2C', 'CAM4']:
    axes[3].plot(compare_t_ms, neuron_data[name][1:], label=f'NEURON {name}')
    axes[3].plot(compare_t_ms[:len(cell_data[name])], cell_data[name], '-.', label=f'BrainCell {name}')
axes[3].set_ylabel('selected CAM* (mM)')
axes[3].legend(ncol=2, fontsize=8)

axes[4].plot(compare_t_ms, neuron_data['pump'][1:], label='NEURON pump')
axes[4].plot(compare_t_ms, neuron_data['pumpca'][1:], label='NEURON pumpca')
axes[4].plot(compare_t_ms[:len(cell_data['pump'])], cell_data['pump'], '-.', label='BrainCell pump')
axes[4].plot(compare_t_ms[:len(cell_data['pumpca'])], cell_data['pumpca'], '-.', label='BrainCell pumpca')
axes[4].plot(compare_t_ms, neuron_total[1:], ':', label='NEURON pump + pumpca')
axes[4].plot(compare_t_ms[:len(cell_total)], cell_total, '--', label='BrainCell pump + pumpca')
axes[4].set_xlabel('time (ms)')
axes[4].set_ylabel('pump states')
axes[4].legend(ncol=2, fontsize=8)

plt.suptitle('CdpStC MA2020 GoC + CaHVA: NEURON vs BrainCell Cell.run')
plt.tight_layout()
plt.show()

print('Per-field Cell.run vs NEURON error summary:')
for name in tracked_fields:
    print(name, error_summary[name])

print('\nPump conserve drift:')
print('  NEURON:', float(np.max(np.abs(neuron_total - neuron_total[0]))))
print('  BrainCell Cell.run:', float(np.max(np.abs(cell_total - cell_total[0]))))

print('\nCa current nonzero check:')
print('  max |-NEURON ica|:', float(np.max(np.abs(neuron_data['ica']))))
print('  max |BrainCell current|:', float(np.max(np.abs(cell_data['ica']))))

cam_extra = ['CAM1N2C', 'CAM1N', 'CAM2N', 'CAM2N1C', 'CAM1C1N']
print('\nExtra CAM* end-state comparison:')
for name in cam_extra:
    print(
        name,
        {
            'NEURON_end': float(neuron_data[name][-1]),
            'cell_end': float(cell_data[name][-1]),
        },
    )
